In [435]:
import argparse
import heapq
import math
import re
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely

from matplotlib.collections import LineCollection
from pyproj import Transformer
from scipy.spatial import cKDTree
from shapely import voronoi_polygons
from shapely.geometry import LineString, MultiPoint, Point, box
from shapely.ops import nearest_points, unary_union
from shapely.plotting import plot_polygon
from shapely.prepared import prep
from shapely.strtree import STRtree

KN = 1.0

In [436]:
def parse_tinggi(v):

    if v is None or (isinstance(v, float) and math.isnan(v)):
        return np.nan

    m = re.search(r"\d+(?:\.\d+)?", str(v).replace(",", "."))

    if m:
        return float(m.group())
    else:
        return np.nan

In [437]:
def kota(rng):

    geoms, hs = [], []

    for i in range(5):
        for j in range(5):
            if rng.random() < 0.12:
                continue
        x0 = 40 + i * 150
        y0 = 40 + j * 150

        geoms.append(box(x0,y0,x0+80,y0+80))
        hs.append(float(rng.choice([20,35,60,90,12,0])))

    gdf = gpd.GeoDataFrame({"h" : hs}, geometry=geoms, crs="EPSG:32748")
    return gdf, (0,0,760,760), None


In [438]:
def load_bangunan(a, rng):

    if a.demo:
        return kota(rng)

    if a.file:
        gdf = gpd.read_file(a.file)
    else:
        import osmnx as ox

        gdf = ox.features_from_point((a.lat, a.lon), tags={"building": True}, dist = a.radius)

    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].reset_index(drop=True)
    gdf = gdf.to_crs(gdf.estimate_utm_crs())
    tr = Transformer.from_crs(4326, gdf.crs, always_xy=True)

    h =  gdf[a.height_col].map(parse_tinggi) if a.height_col in gdf else pd.Series(np.nan, index=gdf.index)

    if "building:levels" in gdf:
        h = h.fillna(pd.to_numeric(gdf["building:levels"], errors="coerce") * 3.2)
    n_unknown = int(h.isna().sum())
    gdf["h"] = h.fillna(a.default_height)
    print(f"Gedung: {len(gdf)} | tinggi tidak diketahui: {n_unknown} (diasumsikan {a.default_height} m)")

    if a.file:
        bounds = tuple(gdf.total_bounds)
    else:
        cx, cy = tr.transform(a.lon, a.lat)
        bounds = (cx - a.radius, cy - a.radius, cx + a.radius, cy + a.radius)
    return gdf, bounds, tr


In [439]:
def make_free_space(bounds, gdf, altitude, margin):
    area = box(*bounds)
    block = gdf[gdf["h"] >= altitude]
    if len(block):
        blocked = unary_union(block.geometry.values).buffer(margin).simplify(0.5)
        area = area.difference(blocked)
    polys = [p for p in getattr(area, "geoms", [area]) if p.area > 200]
    return max(polys, key=lambda p: p.area), len(block)

In [440]:
def snap(free, p):
    inner = free.buffer(-2)
    if inner.is_empty:
        inner = free
    q = nearest_points(inner, Point(p))[0]
    return np.array([q.x, q.y])


In [ ]:
def random_free_point(free, rng):
    minx, miny, maxx, maxy = free.bounds
    inner = prep(free.buffer(-2))
    while True:
        p = (rng.uniform(minx, maxx), rng.uniform(miny, maxy))
        if inner.contains(Point(p)):
            if inner.contains(Point(p)):
                return np.array(p)

In [442]:
def noise(free, n, rng, max_tries=500_000):
    minx, miny, maxx, maxy = free.bounds
    r = 0.7 * math.sqrt(free.area / n)
    pf = prep(free)
    P = np.empty((n, 2))
    k = 0
    for _ in range(max_tries):
        p = np.array([rng.uniform(minx, maxx), rng.uniform(miny, maxy)])
        if not pf.contains(Point(p)):
            continue
        if k and np.min(np.hypot(*(P[:k] - p).T)) < r:
            continue
        P[k] = p
        k += 1
        if k == n:
            break

    return P[:k]

In [443]:
def build_cells(free, seeds):
    vor = voronoi_polygons(MultiPoint([tuple(s) for s in seeds]), extend_to=free)
    polys = list(vor.geoms)
    tree = STRtree(polys)
    cells = []
    kept = []

    for s in seeds:
        hit = tree.query(Point(s), predicate="within")
        if len(hit) == 0:
            continue
        c = polys[hit[0]].intersection(free)
        if c.is_empty:
            continue
        if c.geom_type == "MultiPolygon":
            c = max(c.geoms, key=lambda g: (g.contains(Point(s)), g.area))
        if c.geom_type != "Polygon":
            continue
        cells.append(c)
        kept.append(s)
    return cells, np.array(kept)


In [ ]:
def adjacency(cells):
    tree =  STRtree(cells)
    adj = {i: {} for i in range(len(cells))}

    for i,c in enumerate(cells):
        for j in tree.query(c, predicate="intersects"):
            if j <= i:
                continue
            shared = c.intersection(cells[j])
            if shared.length > 1e-9:
                m = shared.representative_point()
                adj[i][j] = adj[j][i] = (m.x, m.y)
    return adj

In [445]:
class Graph:

    def __init__(self, free, n_seeds, seed=0):
        self.free = free
        self.pfree = prep(free.buffer(0.5))
        self.cells, self.seeds = build_cells(free, noise(free, n_seeds, np.random.default_rng(seed)))

        self.adj = adjacency(self.cells)
        self.tree = STRtree(self.cells)
        self.kd = cKDTree(self.seeds)

    def locate(self, p):
        pt = Point(p)
        hit = self.tree.query(pt, predicate="within")
        return int(hit[0] if len(hit) else int(self.tree.nearest(pt)))

In [446]:
def find_waypoints(g, sog_rng=(2, 20), ncd_thr=10, snsd_rng=(0, 5)):

    if len(g) < 3:
        return np.empty((0, 2))
    x, y, t, cog, sog = (g[c].to_numpy(float) for c in ("x", "y", "t", "cog", "sog"))
 
    ncd = np.abs((cog[1:-1] - cog[:-2] + 180) % 360 - 180)
    dt = np.maximum(t[2:] - t[:-2], 1e-6)
    ns = np.hypot(x[2:] - x[:-2], y[2:] - y[:-2]) / dt
    snsd = np.abs(ns - sog[1:-1] * KN)
    s = sog[1:-1]
 
    ok = ((s >= sog_rng[0]) & (s <= sog_rng[1])
          & (ncd >= ncd_thr)
          & (snsd >= snsd_rng[0]) & (snsd <= snsd_rng[1]))
    return np.c_[x[1:-1][ok], y[1:-1][ok]]
 

In [447]:
def fused_node(center, pts, w_density=1.0, w_center=1.0):
    if pts.empty:
        return center
    density = pts[["x", "y"]].mean().to_numpy()
    num = w_density * density + w_center * center
    den = w_density + w_center
    n_tot = len(pts)
    for _, g in pts.groupby("drone_id"):
        wps = find_waypoints(g.sort_values("t"))
        w = len(g) / n_tot
        if len(wps):
            num = num + w * wps.sum(axis=0)
            den += w * len(wps)
    return num / den

In [448]:
def path_length(pl):
    pl = np.asarray(pl)
    return float(np.hypot(*np.diff(pl, axis=0).T).sum()) if len(pl) > 1 else 0.0

In [449]:
def smooth(path, pfree):
    out, i = [path[0]], 0
    while i < len(path) - 1:
        j = len(path) - 1
        while j > i + 1 and not pfree.covers(LineString([path[i], path[j]])):
            j -= 1
        out.append(path[j])
        i = j
    return np.array(out)

In [450]:
def astar(g, node, keep, s, t):
    goal = node[t]
    cache = {}

    def leg(i, j):
        if (i, j) not in cache:
            a, b = node[i], node[j]
            if g.pfree.covers(LineString([a, b])):
                pl = [a, b]
            else:
                m = np.array(g.adj[i][j])
                ok = g.pfree.covers(LineString([a, m])) and g.pfree.covers(LineString([m, b]))
                pl = [a, m, b] if ok else None
            cache[(i, j)] = None if pl is None else (pl, path_length(pl))
        return cache[(i,j)]

    h = lambda i: float(np.hypot(*(node[i] - goal)))
    openq, cost, parent, closed =  [(h(s), s)], {s: 0.0}, {}, set()

    while openq:
        _, u = heapq.heappop(openq)
        if u in closed:
            continue
        if u == t:
            ids = [u]
            while ids[-1] in parent:
                ids.append(parent[ids[-1]])
            ids.reverse()
            path = [node[ids[0]]]
            for a, b in zip(ids, ids[1:]):
                path += leg(a, b)[0][1:]
            return np.array(path), cost[u], len(closed)
        closed.add(u)
        for v in g.adj[u]:
            if v not in keep or v in closed:
                continue
            lg = leg(u, v)
            if lg is None:
                continue
            ng = cost[u] + lg[1]
            if ng < cost.get(v, math.inf):
                cost[v], parent[v] = ng, u
                heapq.heappush(openq, (ng + h(v), v))
    return None

In [451]:
def tfia_star(g, ais, start, goal, ais_thr=30, traffic_only=True):
    start, goal = np.asarray(start, float), np.asarray(goal, float)
    s, t = g.locate(start), g.locate(goal)
    n = len(g.cells)

    if s == t:
        if g.pfree.covers(LineString([start, goal])):
            return dict(path=np.array([start, goal]), cost=path_length([start, goal]), visited=1, mode="satu sel")
        return None

    if ais is None or len(ais) == 0:
        counts, groups = np.zeros(n, int), {}
    else:
        owner = g.kd.query(ais[["x", "y"]].to_numpy())[1]
        ais = ais.assign(cell=owner)
        counts = np.bincount(owner, minlength=n)
        groups = dict(tuple(ais.groupby("cell")))
    empty = pd.DataFrame(columns=["x", "y", "t", "cog", "sog", "drone_id"])

    traffic = {i for i in range(n) if counts[i] >= ais_thr and len(g.adj[i]) > 1}
    attempts = []

    if traffic_only and traffic:
        attempts.append(("TFIA*", traffic))
    attempts.append(("TFIA* (semua sel)" if traffic else "A* biasa (semua sel)",
                     set(range(n))))

    for label, base in attempts:
        keep = base | {s,t}
        node = {}
        for i in keep - {s, t}:
            p = fused_node(np.array(g.cells[i].centroid.coords[0]), groups.get(i, empty))
            if not g.cells[i].contains(Point(p)):
                q = g.cells[i].representative_point()
                p = np.array([q.x, q.y])
            node[i] = p
        node[s], node[t] = start, goal
        res = astar(g, node, keep, s, t)
        if res is not None:
            path, cost, visited = res
            return dict(path=path, cost=cost, visited=visited, mode=label)
    return None

In [452]:
def bearing(a, b):
    return (math.degrees(math.atan2(b[0] - a[0], b[1] - a[1])) + 360) % 360

In [453]:
def simulate_traffic(g, hubs, rng, n_routes=7, drones_per_route=12):

    rows, did = [], 0
    pairs = [(a, b) for a in range(len(hubs)) for b in range(a + 1, len(hubs))]
    rng.shuffle(pairs)
    for a, b in pairs[:n_routes]:
        res = tfia_star(g, None, hubs[a], hubs[b])
        if res is None:
            continue
        line = LineString(smooth(res["path"], g.pfree))
        L = line.length
        for _ in range(drones_per_route):
            speed = rng.uniform(8, 14)
            d = np.arange(0, L, speed)  # satu sampel per detik
            if len(d) < 4:
                continue
            xy = shapely.get_coordinates(shapely.line_interpolate_point(line, d))
            ahead = shapely.get_coordinates(shapely.line_interpolate_point(line, np.minimum(d + 3, L)))
            back = shapely.get_coordinates(shapely.line_interpolate_point(line, np.maximum(d - 3, 0)))
            cog = np.array([bearing(p, q) for p, q in zip(back, ahead)])
            xy = xy + rng.normal(0, 1.5, xy.shape)
            rows.append(pd.DataFrame({
                "drone_id": did, "t": rng.uniform(0, 3600) + np.arange(len(d)),
                "x": xy[:, 0], "y": xy[:, 1], "cog": cog,
                "sog": speed + rng.normal(0, 0.3, len(d)),
            }))
            did += 1
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=["drone_id", "t", "x", "y", "cog", "sog"])


In [454]:
def obstacle_zones(path, fractions=(0.35, 0.65), half_len=60, r_safe=15):
    line, zones = LineString(path), []
    for f in fractions:
        p = line.interpolate(f, normalized=True)
        a = line.interpolate(max(f - 0.02, 0), normalized=True)
        b = line.interpolate(min(f + 0.02, 1), normalized=True)
        d = np.array([b.x - a.x, b.y - a.y])
        d = d / (np.linalg.norm(d) + 1e-9)
        nrm, c = np.array([-d[1], d[0]]), np.array([p.x, p.y])
        zones.append(LineString([c - nrm * half_len, c + nrm * half_len]).buffer(r_safe))
    return unary_union(zones)


## Visualisasi
Visualisasi dari algoritma

In [455]:
def draw(ax, geom, **kw):
    for part in getattr(geom, "geoms", [geom]):
        plot_polygon(part, ax=ax, add_points=False, **kw)


In [456]:
def plot_all(a, gdf, bounds, free, g, ais, start, goal, res_plain, res_tfia, smooth_path, zones, res_re, out):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor("#e9e9e9")
    draw(ax, free, facecolor="#dceeff", edgecolor="none")
    low = gdf[gdf["h"] < a.altitude]
    high = gdf[gdf["h"] >= a.altitude]
    if len(low):
        low.plot(ax=ax, color="#bdbdbd", zorder=2)
    if len(high):
        high.plot(ax=ax, color="#4a4a4a", zorder=2)
 
    ax.add_collection(LineCollection([np.array(c.exterior.coords) for c in g.cells],
                                     colors="gray", linewidths=0.25, zorder=1))
    if len(ais):
        ax.scatter(ais.x[::4], ais.y[::4], s=1, c="orange", alpha=0.5, zorder=3, label="telemetri drone lain")
    if zones is not None:
        draw(ax, zones, facecolor="#ff9800", edgecolor="#e65100", alpha=0.5, zorder=4)
        ax.plot([], [], color="#ff9800", lw=6, alpha=0.5, label="zona drone obstacle")
    if res_plain:
        ax.plot(*res_plain["path"].T, "--", c="royalblue", lw=1.8, zorder=5,
                label=f"A* biasa ({res_plain['cost']:.0f} m)")
    ax.plot(*res_tfia["path"].T, "-o", c="red", ms=3, lw=2, zorder=6,
            label=f"TFIA* ({res_tfia['cost']:.0f} m)")
    ax.plot(*smooth_path.T, "-", c="black", lw=1.2, zorder=7,
            label=f"TFIA* dihaluskan ({path_length(smooth_path):.0f} m)")
    if res_re:
        ax.plot(*res_re["path"].T, "-", c="limegreen", lw=2.5, zorder=8,
                label=f"rute ulang ({res_re['cost']:.0f} m)")
    ax.scatter(*start, c="green", s=120, zorder=9, label="start")
    ax.scatter(*goal, c="purple", s=180, marker="*", zorder=9, label="goal")

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(f"Navigasi drone TFIA*  |  ketinggian {a.altitude:g} m  |  gedung penghalang abu gelap")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print("Gambar disimpan:", out)
    if not a.no_show:
        plt.show()



## Main
Program Utama

In [457]:
def parse_args():
    p = argparse.ArgumentParser(description="TFIA* untuk drone di ketinggian tetap")
    p.add_argument("--demo", action="store_true", help="kota")
    p.add_argument("--file", help="GeoJSON/Shapefile/GeoParquet gedung (Overture/OSM/lokal)")
    p.add_argument("--lat", type=float, default=-6.1951)
    p.add_argument("--lon", type=float, default=106.8231)
    p.add_argument("--radius", type=float, default=600, help="radius area (m)")
    p.add_argument("--height-col", default="height", help="nama kolom tinggi gedung (meter)")
    p.add_argument("--default-height", type=float, default=15, help="tinggi bila data tidak ada (m)")
    p.add_argument("--altitude", type=float, default=60, help="ketinggian terbang drone (m)")
    p.add_argument("--margin", type=float, default=8, help="margin aman dari gedung (m)")
    p.add_argument("--seeds", type=int, default=1200)
    p.add_argument("--ais-thr", type=int, default=30, help="minimal titik telemetri agar sel dianggap ramai")
    p.add_argument("--start", help="lon,lat (default: pojok barat daya)")
    p.add_argument("--goal", help="lon,lat (default: pojok timur laut)")
    p.add_argument("--obs", type=int, default=2, help="jumlah drone obstacle (0 = nonaktif)")
    p.add_argument("--seed", type=int, default=1)
    p.add_argument("--out", default="tfia_drone.png")
    p.add_argument("--no-show", action="store_true")
    return p.parse_args()

In [458]:
def main():
    a = parse_args()
    rng = np.random.default_rng(a.seed)

    gdf, bounds, tr = load_bangunan(a, rng)
    free, n_block = make_free_space(bounds, gdf, a.altitude, a.margin)
    print(f"Ketinggian {a.altitude:g} m: {n_block} dari {len(gdf)} gedung menghalangi; "
          f"ruang bebas {free.area / 1e4:.1f} ha")
    if n_block == 0:
        print("Tidak ada gedung penghalang; turunkan altitude.")

    minx, miny, maxx, maxy = bounds
    w, h = maxx - minx, maxy - miny
    if a.start and tr:
        start = snap(free, tr.transform(*map(float, a.start.split(","))))
    else:
        start = snap(free, (minx + 0.1 * w, miny + 0.1 * h))
    if a.goal and tr:
        goal = snap(free, tr.transform(*map(float, a.goal.split(","))))
    else:
        goal = snap(free, (maxx - 0.1 * w, maxy - 0.1 * h))

    g = Graph(free, a.seeds)
    print(f"Sel: {len(g.cells)} | sisi: {sum(len(v) for v in g.adj.values()) // 2}")

    res_plain = tfia_star(g, None, start, goal)
    hubs = [start, goal] + [random_free_point(free, rng) for _ in range(4)]
    ais = simulate_traffic(g, hubs, rng)
    print(f"Telemetri simulasi: {len(ais)} titik dari {ais.drone_id.nunique() if len(ais) else 0} drone")

    res = tfia_star(g, ais, start, goal, ais_thr=a.ais_thr)
    if res is None:
        print("TFIA* tidak menemukan rute.")
        return
    sm = smooth(res["path"], g.pfree)

    print(f"\n[{res['mode']}]  biaya {res['cost']:.0f} m | node dikunjungi {res['visited']}")
    if res_plain:
        print(f"[A* biasa]   biaya {res_plain['cost']:.0f} m | node dikunjungi {res_plain['visited']}")
    print(f"[Dihaluskan] panjang {path_length(sm):.0f} m, {len(sm)} titik")

    zones, res_re = None, None
    if a.obs > 0:
        fr = tuple(np.linspace(0.3, 0.7, a.obs))
        zones = obstacle_zones(sm, fr)
        parts = getattr(free.difference(zones), "geoms", [free.difference(zones)])
        free2 = next((p for p in parts if p.buffer(1).covers(Point(start))), None)
        if free2 is not None and free2.buffer(1).covers(Point(goal)):
            g2 = Graph(free2, a.seeds)
            res_re = tfia_star(g2, ais, start, goal, ais_thr=a.ais_thr)
        print("[Rute ulang]", "tidak ada rute" if res_re is None
              else f"biaya {res_re['cost']:.0f} m (menghindari {a.obs} obstacle)")

        plot_all(a, gdf, bounds, free, g, ais, start, goal, res_plain, res, sm, zones, res_re, a.out)


if __name__ == "__main__":
    main()


DataSourceError: '/run/user/1000/jupyter/runtime/kernel-v38de15520cfcab70000f9ad3082f4138046dd6220.json' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.